# Perseptron v2 - 07 Kaggle Submission Generation

Bu notebook H&M yarışma formatına uygun submission dosyaları üretir. Eğitim yapmaz; daha önce eğitilmiş `tabular_only`, `image_history` ve `late_fusion` checkpointlerini kullanarak aynı candidate havuzunu üç modelle skorlar.

Çıktılar:

- `submission_tabular_only.csv`
- `submission_image_history.csv`
- `submission_late_fusion.csv`
- `submission_generation_summary.md`

CNN checkpointi bu notebookta kullanılmaz; CNN bu projede image-only baseline ve Grad-CAM açıklanabilirlik modeli olarak tutulur.

## Ortam, ayarlar ve dosya keşfi

Bu hücre Kaggle input klasörleri içinde gerekli ham veri, embedding cache ve checkpoint dosyalarını arar. Repo içindeki Python modülleri import edilmez; submission üretimi için gereken kod notebook içindedir.

In [ ]:
from pathlib import Path
import json
import os
import time
import warnings

import numpy as np
import pandas as pd
import torch
from torch import nn

warnings.filterwarnings('ignore')

IS_KAGGLE = Path('/kaggle').exists()
WORK_DIR = Path('/kaggle/working') if IS_KAGGLE else Path.cwd()
INPUT_ROOTS = [Path('/kaggle/input')] if IS_KAGGLE else [Path.cwd(), Path.cwd() / 'artifacts' / 'final', Path.cwd() / 'data']

# Smoke modunda format ve path kontrolü hızlı yapılır. Full submission için False yap.
SMOKE_RUN = True
SMOKE_CUSTOMERS = 1000

TOP_K = 12
CANDIDATE_LIMIT = 5000
VISUAL_NEIGHBORS = 3000
CO_PURCHASE_PER_ITEM = 300
BATCH_SIZE = 8192
CUSTOMER_BATCH_SIZE = 128

MODEL_NAMES = ['tabular_only', 'image_history', 'late_fusion']
MODEL_BASENAMES = {
    'tabular_only': ['tabular_only.pt', 'tabular_only_fold0.pt'],
    'image_history': ['image_history.pt', 'image_history_fold0.pt'],
    'late_fusion': ['late_fusion.pt', 'late_fusion_fold0.pt'],
}

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('work_dir:', WORK_DIR)
print('device:', DEVICE)
if torch.cuda.is_available():
    print('cuda devices:', torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

def log(message):
    print(f'[{time.strftime("%H:%M:%S")}] {message}', flush=True)

def find_file(filename, required=True):
    candidates = []
    for root in INPUT_ROOTS:
        if not root.exists():
            continue
        candidates.extend(root.rglob(filename))
    if not candidates and Path(filename).exists():
        candidates.append(Path(filename))
    if not candidates:
        if required:
            raise FileNotFoundError(f'{filename} bulunamadı. Kaggle Add Data inputlarını kontrol et.')
        return None
    # Kaggle'da aynı dosya birden fazla inputta varsa en kısa path genelde en temiz kaynaktır.
    return sorted(candidates, key=lambda p: (len(str(p)), str(p)))[0]

def find_first_file(filenames, required=True):
    missing = []
    for filename in filenames:
        path = find_file(filename, required=False)
        if path is not None:
            return path
        missing.append(filename)
    if required:
        raise FileNotFoundError(f'Hiçbiri bulunamadı: {missing}')
    return None

TRANSACTIONS_PATH = find_file('transactions_train.csv')
CUSTOMERS_PATH = find_file('customers.csv')
ARTICLES_PATH = find_file('articles.csv')
SAMPLE_SUBMISSION_PATH = find_file('sample_submission.csv')
EMBEDDINGS_PATH = find_file('article_image_embeddings_popular.npy')
EMBEDDING_IDS_PATH = find_file('article_image_embedding_ids_popular.csv')
MODEL_PATHS = {name: find_first_file(filenames) for name, filenames in MODEL_BASENAMES.items()}

print('transactions:', TRANSACTIONS_PATH)
print('customers:', CUSTOMERS_PATH)
print('articles:', ARTICLES_PATH)
print('sample_submission:', SAMPLE_SUBMISSION_PATH)
print('embeddings:', EMBEDDINGS_PATH)
print('embedding ids:', EMBEDDING_IDS_PATH)
print('models:', MODEL_PATHS)


## Model sınıfları ve feature yardımcıları

Bu hücre checkpointlerdeki mimariyi yeniden kurar, metadata ile tabular encoding yapar ve image-history / late-fusion girdilerini hazırlar.

In [ ]:
def embedding_dim(size: int) -> int:
    return min(50, max(4, int(size**0.25 * 8)))

class TabularOnlyMLP(nn.Module):
    def __init__(self, numeric_dim: int, category_sizes: list[int]):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(size, embedding_dim(size)) for size in category_sizes])
        cat_dim = sum(embedding.embedding_dim for embedding in self.embeddings)
        self.net = nn.Sequential(
            nn.Linear(numeric_dim + cat_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, 1),
        )

    def forward(self, numeric, categorical):
        embedded = [emb(categorical[:, idx]) for idx, emb in enumerate(self.embeddings)]
        return self.net(torch.cat([numeric, *embedded], dim=1)).squeeze(1)

class ImageHistoryMLP(nn.Module):
    def __init__(self, image_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(image_dim * 2 + 2, 512), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(512, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, 1),
        )

    def forward(self, article_embedding, profile_embedding, visual_similarity, visual_history_count):
        visual_extra = torch.stack([visual_similarity, visual_history_count], dim=1)
        return self.net(torch.cat([article_embedding, profile_embedding, visual_extra], dim=1)).squeeze(1)

class MultimodalLateFusion(nn.Module):
    def __init__(self, numeric_dim: int, category_sizes: list[int], image_dim: int):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(size, embedding_dim(size)) for size in category_sizes])
        cat_dim = sum(embedding.embedding_dim for embedding in self.embeddings)
        self.tabular_branch = nn.Sequential(
            nn.Linear(numeric_dim + cat_dim, 256), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(256, 128), nn.ReLU(),
        )
        self.visual_branch = nn.Sequential(
            nn.Linear(image_dim * 2 + 2, 512), nn.ReLU(), nn.Dropout(0.25),
            nn.Linear(512, 128), nn.ReLU(),
        )
        self.fusion_head = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.1), nn.Linear(128, 1))

    def forward(self, numeric, categorical, article_embedding, profile_embedding, visual_similarity, visual_history_count):
        embedded = [emb(categorical[:, idx]) for idx, emb in enumerate(self.embeddings)]
        tabular = self.tabular_branch(torch.cat([numeric, *embedded], dim=1))
        visual_extra = torch.stack([visual_similarity, visual_history_count], dim=1)
        visual = self.visual_branch(torch.cat([article_embedding, profile_embedding, visual_extra], dim=1))
        return self.fusion_head(torch.cat([tabular, visual], dim=1)).squeeze(1)

def build_model(model_name: str, metadata: dict, image_dim: int):
    category_sizes = [len(metadata['category_maps'][column]) for column in metadata['categorical_features']]
    numeric_dim = len(metadata['numeric_features'])
    if model_name == 'tabular_only':
        return TabularOnlyMLP(numeric_dim, category_sizes)
    if model_name == 'image_history':
        return ImageHistoryMLP(image_dim)
    if model_name == 'late_fusion':
        return MultimodalLateFusion(numeric_dim, category_sizes, image_dim)
    raise ValueError(f'Unknown model: {model_name}')

def load_checkpoint_model(path: Path, device: torch.device):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model = build_model(checkpoint['model_name'], checkpoint['metadata'], checkpoint['image_dim']).to(device)
    model.load_state_dict(checkpoint['state_dict'])
    model.eval()
    return checkpoint, model

def encode_tabular(frame: pd.DataFrame, metadata: dict):
    numeric_columns = []
    for column in metadata['numeric_features']:
        values = pd.to_numeric(frame[column], errors='coerce').fillna(metadata['numeric_mean'][column]).astype('float32')
        values = (values - metadata['numeric_mean'][column]) / metadata['numeric_std'][column]
        numeric_columns.append(values.to_numpy(dtype='float32'))
    numeric = np.stack(numeric_columns, axis=1).astype('float32')

    categorical_columns = []
    for column in metadata['categorical_features']:
        mapping = metadata['category_maps'][column]
        values = frame[column].fillna('UNKNOWN').astype(str).map(mapping).fillna(0).astype('int64')
        categorical_columns.append(values.to_numpy(dtype='int64'))
    categorical = np.stack(categorical_columns, axis=1).astype('int64')
    return numeric, categorical

def forward_model(model_name, model, batch, device):
    if model_name == 'tabular_only':
        return model(batch['numeric'].to(device), batch['categorical'].to(device))
    if model_name == 'image_history':
        return model(
            batch['article_emb'].to(device), batch['profile_emb'].to(device),
            batch['visual_similarity'].to(device), batch['visual_history_count'].to(device),
        )
    if model_name == 'late_fusion':
        return model(
            batch['numeric'].to(device), batch['categorical'].to(device),
            batch['article_emb'].to(device), batch['profile_emb'].to(device),
            batch['visual_similarity'].to(device), batch['visual_history_count'].to(device),
        )
    raise ValueError(model_name)


## Veri yükleme ve ortak candidate kaynakları

Bu hücre full training history üzerinden global popular, co-purchase ve customer visual profile yapılarını hazırlar. `sample_submission.csv` yarışma formatındaki nihai customer listesini belirler.

In [ ]:
def normalize_article_id_series(series):
    return series.astype(str).str.replace(r'\.0$', '', regex=True).str.zfill(10)

log('Ham tablolar okunuyor...')
transactions = pd.read_csv(
    TRANSACTIONS_PATH,
    dtype={'customer_id': 'string', 'article_id': 'string'},
    usecols=['customer_id', 'article_id', 't_dat'],
)
transactions['article_id'] = normalize_article_id_series(transactions['article_id'])
transactions['customer_id'] = transactions['customer_id'].astype(str)
transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

customers = pd.read_csv(CUSTOMERS_PATH, dtype={'customer_id': 'string'})
customers['customer_id'] = customers['customer_id'].astype(str)
articles = pd.read_csv(ARTICLES_PATH, dtype={'article_id': 'string'})
articles['article_id'] = normalize_article_id_series(articles['article_id'])
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH, dtype={'customer_id': 'string'})
sample_submission['customer_id'] = sample_submission['customer_id'].astype(str)

if SMOKE_RUN:
    sample_submission = sample_submission.head(SMOKE_CUSTOMERS).copy()
    log(f'Smoke mode aktif: {len(sample_submission):,} customer işlenecek.')
else:
    log(f'Full mode aktif: {len(sample_submission):,} customer işlenecek.')

log('Embedding cache okunuyor...')
embeddings = np.load(EMBEDDINGS_PATH, mmap_mode=None).astype('float32')
embedding_ids = pd.read_csv(EMBEDDING_IDS_PATH, dtype={'article_id': 'string'})
article_ids = normalize_article_id_series(embedding_ids['article_id']).tolist()
article_to_index = {article_id: index for index, article_id in enumerate(article_ids)}
embedding_norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
embeddings_normalized = embeddings / np.maximum(embedding_norms, 1e-8)

log('Global popular listesi hazırlanıyor...')
global_popular = transactions['article_id'].value_counts().head(max(CANDIDATE_LIMIT, TOP_K)).index.astype(str).tolist()

log('Co-purchase haritası hazırlanıyor...')
recent_transactions = transactions.sort_values('t_dat').groupby('customer_id').tail(50)
baskets = recent_transactions.groupby('customer_id')['article_id'].apply(lambda values: list(dict.fromkeys(values))).tolist()
co_counts = {}
for basket in baskets:
    unique = basket[-50:]
    for item in unique:
        target = co_counts.setdefault(item, {})
        for other in unique:
            if other != item:
                target[other] = target.get(other, 0) + 1
co_purchase = {
    item: [other for other, _ in sorted(others.items(), key=lambda kv: kv[1], reverse=True)[:CO_PURCHASE_PER_ITEM]]
    for item, others in co_counts.items()
}
del co_counts, baskets

log('Customer history ve visual profile hazırlanıyor...')
history = transactions[['customer_id', 'article_id']].copy()
history_by_customer = history.groupby('customer_id')['article_id'].apply(list).to_dict()
profile_sums = {}
profile_counts = {}
for customer_id, items in history_by_customer.items():
    indices = [article_to_index[item] for item in items if item in article_to_index]
    if not indices:
        continue
    profile_sums[customer_id] = embeddings[indices].sum(axis=0).astype('float32')
    profile_counts[customer_id] = float(len(indices))

models = {}
for name, path in MODEL_PATHS.items():
    checkpoint, model = load_checkpoint_model(path, DEVICE)
    models[name] = (checkpoint, model)
    log(f'Model yüklendi: {name} -> {path.name}')

log('Veri hazırlığı tamamlandı.')
print('transactions:', transactions.shape)
print('customers:', customers.shape)
print('articles:', articles.shape)
print('sample_submission:', sample_submission.shape)
print('embedding matrix:', embeddings.shape)


## Candidate üretimi ve model skorlaması

Bu hücre her customer için aday ürün listesi kurar, üç modeli aynı aday havuzu üzerinde çalıştırır ve top-12 prediction üretir.

In [ ]:
def merge_metadata(pairs):
    return pairs.merge(customers, on='customer_id', how='left').merge(articles, on='article_id', how='left')

def visual_arrays_for_pairs(pairs, profile, profile_count):
    image_dim = embeddings.shape[1]
    article_emb = np.zeros((len(pairs), image_dim), dtype='float32')
    profile_emb = np.zeros((len(pairs), image_dim), dtype='float32')
    visual_similarity = np.zeros(len(pairs), dtype='float32')
    visual_history_count = np.zeros(len(pairs), dtype='float32')
    if profile is None:
        return article_emb, profile_emb, visual_similarity, visual_history_count
    profile_norm = float(np.linalg.norm(profile))
    if profile_norm <= 0:
        return article_emb, profile_emb, visual_similarity, visual_history_count
    for index, article_id in enumerate(pairs['article_id'].astype(str)):
        article_idx = article_to_index.get(article_id)
        if article_idx is None:
            continue
        candidate = embeddings[article_idx]
        article_emb[index] = candidate
        profile_emb[index] = profile
        visual_history_count[index] = float(profile_count or 0.0)
        denom = np.linalg.norm(candidate) * profile_norm
        visual_similarity[index] = float(np.dot(candidate, profile) / denom) if denom > 0 else 0.0
    return article_emb, profile_emb, visual_similarity, visual_history_count

def visual_neighbors_for_profile(profile):
    if profile is None or np.linalg.norm(profile) <= 0 or VISUAL_NEIGHBORS <= 0:
        return []
    profile_unit = profile / max(np.linalg.norm(profile), 1e-8)
    scores = embeddings_normalized @ profile_unit.astype('float32')
    take = min(VISUAL_NEIGHBORS, len(scores))
    if take <= 0:
        return []
    top_indices = np.argpartition(-scores, take - 1)[:take]
    top_indices = top_indices[np.argsort(-scores[top_indices])]
    return [article_ids[index] for index in top_indices]

def candidate_pool_for_customer(customer_id):
    history_items = [str(item) for item in history_by_customer.get(customer_id, [])]
    profile = None
    profile_count = 0.0
    if customer_id in profile_sums:
        profile_count = profile_counts.get(customer_id, 0.0)
        profile = profile_sums[customer_id] / max(profile_count, 1.0)

    candidates = []
    candidates.extend(global_popular[:min(CANDIDATE_LIMIT, 1000)])
    for item in history_items[-20:]:
        candidates.extend(co_purchase.get(item, [])[:50])
    candidates.extend(visual_neighbors_for_profile(profile))

    seen = set(history_items)
    output = []
    for article_id in candidates:
        article_id = str(article_id).zfill(10)
        if article_id in seen:
            continue
        seen.add(article_id)
        output.append(article_id)
        if len(output) >= CANDIDATE_LIMIT:
            break
    for article_id in global_popular:
        if len(output) >= TOP_K:
            break
        if article_id not in seen:
            seen.add(article_id)
            output.append(article_id)
    return output[:CANDIDATE_LIMIT], profile, profile_count

def score_pairs(checkpoint, model, pairs, frame, visual_arrays):
    model_name = checkpoint['model_name']
    article_emb, profile_emb, visual_similarity, visual_history_count = visual_arrays
    if model_name in {'tabular_only', 'late_fusion'}:
        numeric, categorical = encode_tabular(frame, checkpoint['metadata'])
    else:
        numeric = categorical = None

    scores = []
    with torch.no_grad():
        for start in range(0, len(pairs), BATCH_SIZE):
            end = min(start + BATCH_SIZE, len(pairs))
            batch = {}
            if model_name in {'tabular_only', 'late_fusion'}:
                batch['numeric'] = torch.tensor(numeric[start:end], dtype=torch.float32)
                batch['categorical'] = torch.tensor(categorical[start:end], dtype=torch.long)
            if model_name in {'image_history', 'late_fusion'}:
                batch['article_emb'] = torch.tensor(article_emb[start:end], dtype=torch.float32)
                batch['profile_emb'] = torch.tensor(profile_emb[start:end], dtype=torch.float32)
                batch['visual_similarity'] = torch.tensor(visual_similarity[start:end], dtype=torch.float32)
                batch['visual_history_count'] = torch.tensor(visual_history_count[start:end], dtype=torch.float32)
            logits = forward_model(model_name, model, batch, DEVICE)
            scores.append(torch.sigmoid(logits).detach().cpu().numpy())
    return np.concatenate(scores) if scores else np.array([], dtype='float32')

def rank_customer(customer_id):
    candidates, profile, profile_count = candidate_pool_for_customer(customer_id)
    if not candidates:
        fallback = global_popular[:TOP_K]
        return {name: fallback for name in MODEL_NAMES}
    pairs = pd.DataFrame({'customer_id': customer_id, 'article_id': candidates})
    frame = merge_metadata(pairs)
    visual = visual_arrays_for_pairs(pairs, profile, profile_count)
    ranked = {}
    for name, (checkpoint, model) in models.items():
        scores = score_pairs(checkpoint, model, pairs, frame, visual)
        order = np.argsort(-scores)
        prediction = [candidates[index] for index in order[:TOP_K]]
        if len(prediction) < TOP_K:
            prediction += [item for item in global_popular if item not in prediction][:TOP_K - len(prediction)]
        ranked[name] = prediction[:TOP_K]
    return ranked


## Submission dosyalarını üret ve doğrula

Bu hücre üç model için ayrı CSV üretir. Kaggle submission kolonları yalnızca `customer_id` ve `prediction` olarak tutulur.

In [ ]:
rows = {name: [] for name in MODEL_NAMES}
start_time = time.time()
customer_ids = sample_submission['customer_id'].astype(str).tolist()

for index, customer_id in enumerate(customer_ids, start=1):
    ranked = rank_customer(customer_id)
    for name in MODEL_NAMES:
        rows[name].append({'customer_id': customer_id, 'prediction': ' '.join(ranked[name])})
    if index % 100 == 0 or index == len(customer_ids):
        elapsed = time.time() - start_time
        log(f'{index:,}/{len(customer_ids):,} customer tamamlandı | elapsed={elapsed/60:.1f} dk')

summary_lines = [
    '# Submission Generation Summary',
    '',
    f'- Smoke run: {SMOKE_RUN}',
    f'- Customers: {len(customer_ids):,}',
    f'- Top-k: {TOP_K}',
    f'- Candidate limit: {CANDIDATE_LIMIT}',
    f'- Visual neighbors: {VISUAL_NEIGHBORS}',
    f'- Co-purchase per item: {CO_PURCHASE_PER_ITEM}',
    f'- Device: {DEVICE}',
    '',
    '| model | output | rows | valid_top12 |',
    '| --- | --- | ---: | --- |',
]

for name in MODEL_NAMES:
    output = pd.DataFrame(rows[name])
    output = sample_submission[['customer_id']].merge(output, on='customer_id', how='left')
    output['prediction'] = output['prediction'].fillna(' '.join(global_popular[:TOP_K]))
    output_path = WORK_DIR / f'submission_{name}.csv'
    output.to_csv(output_path, index=False)
    valid_top12 = output['prediction'].astype(str).str.split().map(len).eq(TOP_K).all()
    if list(output.columns) != ['customer_id', 'prediction']:
        raise AssertionError(f'{name}: kolon formatı hatalı')
    if len(output) != len(sample_submission):
        raise AssertionError(f'{name}: satır sayısı sample_submission ile eşleşmiyor')
    if not valid_top12:
        raise AssertionError(f'{name}: bazı prediction satırları {TOP_K} ürün içermiyor')
    summary_lines.append(f'| {name} | `{output_path.name}` | {len(output):,} | {valid_top12} |')
    log(f'Kaydedildi: {output_path}')

summary_path = WORK_DIR / 'submission_generation_summary.md'
summary_path.write_text('\n'.join(summary_lines) + '\n', encoding='utf-8')
print('\n'.join(summary_lines))
log(f'Özet kaydedildi: {summary_path}')


## Kaggle submit komutları

Full run tamamlandıktan sonra output CSV dosyaları ayrı ayrı submit edilebilir. Önce `late_fusion` gönderilir; skor dönerse karşılaştırma için diğer iki baseline submission da gönderilir.

```bash
kaggle competitions submit -c h-and-m-personalized-fashion-recommendations -f /kaggle/working/submission_late_fusion.csv -m "perseptron late_fusion full customer"
kaggle competitions submit -c h-and-m-personalized-fashion-recommendations -f /kaggle/working/submission_tabular_only.csv -m "perseptron tabular_only full customer"
kaggle competitions submit -c h-and-m-personalized-fashion-recommendations -f /kaggle/working/submission_image_history.csv -m "perseptron image_history full customer"
```